# Phase 06 — Modeling: Support Vector Machine (SVM)

**Học phần:** CS114 — Machine Learning (ThS. Cáp Phạm Đình Thăng)

**Thành viên phụ trách:** **Bùi Thị Mỷ Cẩm** (MSSV: `25730013`)

**Mục tiêu:**
- Nạp ma trận đặc trưng Zero-Leakage từ Phase 03 (`X_train.pkl`, `X_test.pkl`: 5.015 chiều).
- Huấn luyện `LinearSVC` bọc trong `CalibratedClassifierCV` (để có `predict_proba`), bù mất cân bằng lớp `class_weight='balanced'`.
- So sánh đối chứng nhanh với SVM nhân RBF (`kernel='rbf'`).
- Tinh chỉnh siêu tham số `C` bằng `GridSearchCV` (5-fold CV).
- Đánh giá toàn diện trên tập kiểm thử độc lập `X_test` (439 mẫu), đo 5 chỉ số (Accuracy, Precision, Recall, F1, ROC-AUC).
- Vẽ biểu đồ học thuật 300 DPI: Confusion Matrix, ROC Curve, Learning Curve, Top-20 |Coefficient| Weight.
- Đóng gói **`models/best_model_bundle.pkl`** — hợp đồng suy luận cứng cho Phase 08 (Demo UI — Sơn).

> **CRITICAL INVARIANT:** KHÔNG fit lại TF-IDF hay split lại dataset — dùng thẳng `X_train.pkl` / `X_test.pkl` từ Phase 03 để đảm bảo tính nhất quán với Phase 04, 05 và Phase 07.

## 0. Setup & Imports

In [ ]:
import json
import sys
import time
from pathlib import Path

import joblib
import numpy as np
import sklearn
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.svm import SVC, LinearSVC

PROJECT_ROOT = Path('.').resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.feature_engineering_config import AUX_FEATURE_COLS
from src.model_plots import (
    compute_metrics,
    plot_confusion_matrix,
    plot_feature_importance,
    plot_learning_curve_chart,
    plot_roc_curve,
)
from src.preprocessing.pipeline import preprocess_pipeline
from src.utils.helpers import RANDOM_STATE

DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
MODELS_DIR = PROJECT_ROOT / 'models'
FIGURES_DIR = PROJECT_ROOT / 'reports' / 'figures'

MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print('Setup completed successfully!')
print(f'RANDOM_STATE = {RANDOM_STATE}, sklearn version = {sklearn.__version__}')

## 1. Nạp Ma trận Đặc trưng Zero-Leakage từ Phase 03

In [ ]:
X_train = joblib.load(DATA_DIR / 'X_train.pkl')   # (1754, 5015) sparse CSR
X_test  = joblib.load(DATA_DIR / 'X_test.pkl')    # (439, 5015) sparse CSR
y_train = joblib.load(DATA_DIR / 'y_train.pkl')   # (1754,)
y_test  = joblib.load(DATA_DIR / 'y_test.pkl')    # (439,)

with open(DATA_DIR / 'feature_names.json', encoding='utf-8') as f:
    feature_names = json.load(f)

# Vectorizer & aux scaler fit sẵn ở Phase 03 — tái sử dụng nguyên trạng cho bundle suy luận
tfidf_vectorizer = joblib.load(DATA_DIR / 'tfidf_vectorizer.pkl')
aux_scaler = joblib.load(DATA_DIR / 'aux_scaler.pkl')

print(f'X_train shape: {X_train.shape} | y_train shape: {y_train.shape}')
print(f'X_test shape:  {X_test.shape}  | y_test shape:  {y_test.shape}')
print(f'Feature space dimensionality: {len(feature_names)}')
print(f'Train labels distribution: Normal={(y_train == 0).sum()} ({((y_train == 0).mean()*100):.1f}%), Scam={(y_train == 1).sum()} ({((y_train == 1).mean()*100):.1f}%)')
print(f'Test labels distribution:  Normal={(y_test == 0).sum()} ({((y_test == 0).mean()*100):.1f}%), Scam={(y_test == 1).sum()} ({((y_test == 1).mean()*100):.1f}%)')

In [ ]:
# Tách tập Validation nội bộ (15% của X_train) để đánh giá nhanh baseline
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.15,
    random_state=RANDOM_STATE, stratify=y_train,
)
print(f'X_tr: {X_tr.shape}, X_val: {X_val.shape}')

## 2. Baseline SVM Linear (LinearSVC + CalibratedClassifierCV để có predict_proba)

In [ ]:
svm_base = LinearSVC(
    C=1.0,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    max_iter=5000,
)
svm_baseline = CalibratedClassifierCV(svm_base, cv=5)

t0 = time.time()
svm_baseline.fit(X_tr, y_tr)
print(f'SVM Linear Baseline trained in {time.time()-t0:.2f}s')

y_val_pred = svm_baseline.predict(X_val)
y_val_proba = svm_baseline.predict_proba(X_val)[:, 1]
baseline_val_metrics = compute_metrics(y_val, y_val_pred, y_val_proba)

print('--- SVM Linear Baseline (Validation Set) ---')
for k, v in baseline_val_metrics.items():
    print(f'  {k:12s}: {v:.4f}')

## 3. SVM Nhân RBF (Đối chứng — Alternative Kernel)

In [ ]:
# ⚠️ RBF chậm hơn Linear trên ma trận thưa lớn — chỉ chạy đối chứng nhanh trên X_tr/X_val,
# KHÔNG dùng để chọn model cuối cùng (Linear vẫn là lựa chọn mặc định theo Phase 06 plan).
svm_rbf = SVC(
    kernel='rbf',
    C=1.0,
    gamma='scale',
    class_weight='balanced',
    random_state=RANDOM_STATE,
    probability=True,
)

t0 = time.time()
svm_rbf.fit(X_tr, y_tr)
rbf_fit_time = time.time() - t0
print(f'SVM RBF trained in {rbf_fit_time:.2f}s')

y_val_pred_rbf = svm_rbf.predict(X_val)
y_val_proba_rbf = svm_rbf.predict_proba(X_val)[:, 1]
rbf_val_metrics = compute_metrics(y_val, y_val_pred_rbf, y_val_proba_rbf)

print('--- SVM RBF (Validation Set) ---')
for k, v in rbf_val_metrics.items():
    print(f'  {k:12s}: {v:.4f}')
print(f'\nLinear F1={baseline_val_metrics["f1"]:.4f} vs RBF F1={rbf_val_metrics["f1"]:.4f} '
      f'-> giữ Linear (nhanh hơn, phù hợp dữ liệu thưa TF-IDF, dễ diễn giải hệ số).')

## 4. Tinh chỉnh Siêu tham số — GridSearchCV trên C (5-Fold CV trên X_train)

In [ ]:
param_grid = {
    'estimator__C': [0.01, 0.1, 1.0, 10.0],
}

grid_svm = GridSearchCV(
    CalibratedClassifierCV(
        LinearSVC(class_weight='balanced', random_state=RANDOM_STATE, max_iter=5000),
    ),
    param_grid, cv=5, scoring='f1', n_jobs=-1,
)

t0 = time.time()
print('Đang thực thi GridSearchCV (4 giá trị C, 5-Fold CV = 20 fits)...')
grid_svm.fit(X_train, y_train)
print(f'GridSearchCV hoàn tất trong {time.time()-t0:.1f}s')
print(f'Best Parameters: {grid_svm.best_params_}')
print(f'Best 5-Fold CV F1 Score: {grid_svm.best_score_:.4f}')

## 5. Huấn luyện Model Tối ưu & Đánh giá trên Test Set Độc lập (439 mẫu)

In [ ]:
# Trích xuất model tốt nhất (đã được refit tự động trên toàn bộ X_train bởi GridSearchCV)
svm_model = grid_svm.best_estimator_

y_test_pred = svm_model.predict(X_test)
y_test_proba = svm_model.predict_proba(X_test)[:, 1]
test_metrics = compute_metrics(y_test, y_test_pred, y_test_proba)

print('======================================================')
print('         KẾT QUẢ ĐÁNH GIÁ SVM LINEAR TRÊN TEST SET   ')
print('======================================================')
for k, v in test_metrics.items():
    print(f'  {k:12s}: {v:.4f}')
print('------------------------------------------------------')
print(classification_report(y_test, y_test_pred, target_names=['Bình thường (0)', 'Lừa đảo (1)'], digits=4))

## 6. Trực quan hóa Ma trận Nhầm lẫn & Đường cong ROC

In [ ]:
plot_confusion_matrix(
    y_test, y_test_pred, FIGURES_DIR, 'svm_confusion_matrix.png',
    'Confusion Matrix — SVM Linear (Test Set)',
)

auc_score = plot_roc_curve(
    y_test, y_test_proba, FIGURES_DIR, 'svm_roc_curve.png',
    'ROC Curve — SVM Linear (Test Set)',
)
print(f'Biểu đồ đã lưu: svm_confusion_matrix.png & svm_roc_curve.png (AUC = {auc_score:.4f})')

## 7. Đánh giá Overfitting qua Đường cong Học tập (Learning Curve)

In [ ]:
best_C = grid_svm.best_params_['estimator__C']
plot_learning_curve_chart(
    CalibratedClassifierCV(
        LinearSVC(C=best_C, class_weight='balanced', random_state=RANDOM_STATE, max_iter=5000),
    ),
    X_train, y_train, FIGURES_DIR, 'svm_learning_curve.png',
    'Learning Curve — SVM Linear (tuned)', cv=5, scoring='f1', random_state=RANDOM_STATE,
)
print('Biểu đồ đã lưu: svm_learning_curve.png')

## 8. Trích xuất & Phân tích Top-20 Đặc trưng Có Trọng số Lớn Nhất (|Coefficient|)

In [ ]:
# LinearSVC là mô hình tuyến tính — |hệ số| lớn nghĩa là đặc trưng đóng góp mạnh vào quyết định.
# Lấy trung bình |coef_| trên 5 folds calibration để ổn định hơn 1 fold đơn lẻ.
fold_coefs = np.array([
    cc.estimator.coef_.ravel() for cc in svm_model.calibrated_classifiers_
])
mean_abs_coefs = np.abs(fold_coefs).mean(axis=0)

plot_feature_importance(
    mean_abs_coefs, feature_names, FIGURES_DIR, 'svm_feature_importance.png',
    'Top-20 Đặc trưng |Trọng số| Lớn Nhất — SVM Linear',
    top_n=20,
)

top_idx = np.argsort(mean_abs_coefs)[::-1][:20]
print('Top-20 đặc trưng có |hệ số| lớn nhất của SVM Linear:')
for rank, idx in enumerate(top_idx, 1):
    print(f'  {rank:2d}. {feature_names[idx]:32s} |coef|: {mean_abs_coefs[idx]:.4f}')

## 9. Đóng gói & Lưu trữ Model (.pkl) và Hồ sơ Chỉ số (.json)

In [ ]:
# 1. Lưu trọng số mô hình (đúng đường dẫn quy định trong Phase 06 plan)
model_file = MODELS_DIR / 'svm_linear.pkl'
joblib.dump(svm_model, model_file)

# 2. Lưu hồ sơ chỉ số kỹ thuật
metrics_record = {
    'algorithm': 'SVM Linear (LinearSVC + CalibratedClassifierCV)',
    'author': 'Bùi Thị Mỷ Cẩm (MSSV: 25730013)',
    'phase': '06 — Modeling (SVM)',
    'best_params': grid_svm.best_params_,
    'cv_f1_gridsearch': float(grid_svm.best_score_),
    'baseline_val_metrics': baseline_val_metrics,
    'rbf_kernel_val_metrics_comparison': rbf_val_metrics,
    'test_metrics': test_metrics,
    'sklearn_version': sklearn.__version__,
    'random_state': RANDOM_STATE,
    'n_features': len(feature_names),
}

metrics_file = MODELS_DIR / 'svm_metrics.json'
with open(metrics_file, 'w', encoding='utf-8') as f:
    json.dump(metrics_record, f, ensure_ascii=False, indent=2)

import os
model_size_mb = os.path.getsize(model_file) / (1024 * 1024)
print(f'✅ Model saved:   {model_file} ({model_size_mb:.2f} MB)')
print(f'✅ Metrics saved: {metrics_file}')
print(json.dumps(metrics_record, ensure_ascii=False, indent=2))

## 10. Đóng gói Hợp đồng Suy luận — `models/best_model_bundle.pkl` (Phase 06 → Phase 08)

> [!IMPORTANT]
> Phase 08 (Demo UI — Sơn) phụ thuộc trực tiếp vào định dạng chính xác của file này. Bundle gộp `model` + `vectorizer` + `aux_scaler` + `feature_names` để suy luận độc lập trên văn bản thô, không cần refit gì thêm.

In [ ]:
best_model_bundle = {
    'model': svm_model,
    'vectorizer': tfidf_vectorizer,
    'aux_scaler': aux_scaler,
    'feature_names': feature_names,
    'model_name': 'SVM Linear',
    'threshold': 0.5,
    'metrics': {
        'f1': test_metrics['f1'],
        'recall_scam': test_metrics['recall'],
        'precision': test_metrics['precision'],
        'roc_auc': test_metrics['roc_auc'],
    },
    'metadata': {
        'algorithm': 'Support Vector Machine (LinearSVC)',
        'random_state': RANDOM_STATE,
        'sklearn_version': sklearn.__version__,
    },
}

bundle_file = MODELS_DIR / 'best_model_bundle.pkl'
joblib.dump(best_model_bundle, bundle_file)
bundle_size_mb = os.path.getsize(bundle_file) / (1024 * 1024)
print(f'✅ Bundle saved: {bundle_file} ({bundle_size_mb:.2f} MB)')

## 11. Kiểm thử Hợp đồng Suy luận End-to-End (Raw Text → Prediction)

Mô phỏng chính xác luồng suy luận mà Phase 08 (Streamlit) sẽ dùng: tiền xử lý văn bản thô, trích 15 đặc trưng phụ, chuẩn hóa, ghép TF-IDF, và dự đoán — để phát hiện sớm mọi lỗi lệch chiều không gian đặc trưng trước khi bàn giao.

In [ ]:
from scipy.sparse import csr_matrix, hstack
import pandas as pd

def predict_raw_text(text: str, bundle: dict) -> dict:
    cleaned_text, aux_dict = preprocess_pipeline(text)
    aux_df = pd.DataFrame([aux_dict])[AUX_FEATURE_COLS]
    aux_scaled = bundle['aux_scaler'].transform(aux_df)
    tfidf_vec = bundle['vectorizer'].transform([cleaned_text])
    full_vec = hstack([tfidf_vec, csr_matrix(aux_scaled)])
    proba = float(bundle['model'].predict_proba(full_vec)[0][1])
    label = 1 if proba >= bundle.get('threshold', 0.5) else 0
    return {'label': label, 'risk_score': proba}

sample_scam = 'Chào ông/bà, chúng tôi là công an điều tra, tài khoản của ông/bà liên quan đường dây rửa tiền, chuyển khoản ngay 50 triệu để xác minh kẻo bị bắt giam'
sample_normal = 'Ngân hàng thông báo số dư tài khoản của bạn vào ngày hôm nay là 3.250.000 đồng. Cảm ơn quý khách đã sử dụng dịch vụ.'

loaded_bundle = joblib.load(MODELS_DIR / 'best_model_bundle.pkl')
for label_txt, txt in [('SCAM mẫu', sample_scam), ('NORMAL mẫu', sample_normal)]:
    result = predict_raw_text(txt, loaded_bundle)
    print(f"[{label_txt}] label={result['label']} ({'Lừa đảo' if result['label'] == 1 else 'Bình thường'}), risk_score={result['risk_score']:.4f}")

## 12. Tổng kết Thực nghiệm Phase 06 (SVM)

- **Hoàn thành:** Huấn luyện SVM Linear (Calibrated) với siêu tham số C tối ưu trên không gian 5.015 chiều Zero-Leakage, đối chứng với SVM RBF.
- **Chỉ số đạt được:** Xem `models/svm_metrics.json` — so sánh trực tiếp với Naive Bayes (Phase 04), XGBoost & Random Forest (Phase 05).
- **Hợp đồng bàn giao:** `models/best_model_bundle.pkl` đã đóng gói đầy đủ model + vectorizer + aux_scaler + feature_names, kiểm thử end-to-end thành công trên văn bản thô.
- **Sẵn sàng:** Phase 07 (So sánh chéo 4 mô hình & XAI — Thanh) và Phase 08 (Demo UI Streamlit — Sơn).
- **PhoBERT (Optional):** Không thực hiện trong sprint này do giới hạn thời gian/GPU trước hạn bảo vệ 23/09 — có thể bổ sung sau nếu còn dư thời gian.